In [0]:
# ================================================================
# PHASE 22 — PIPELINE SCHEDULING / JOB ORCHESTRATION
# ================================================================

print("=" * 70)
print("PHASE 22 — PIPELINE SCHEDULING / JOB ORCHESTRATION")
print("=" * 70)

print()
print("Purpose:")
print("Validate the pipeline execution order and scheduling configuration.")
print()
print("Pipeline:")
print("Bronze → Silver → Gold → Data Quality → RAG → Copilot")

In [0]:
# ================================================================
# CELL 2 — IMPORT DEPENDENCIES
# ================================================================

import json
import time
from datetime import datetime

print("=" * 70)
print("LOADING DEPENDENCIES")
print("=" * 70)

print("Imports successful.")

In [0]:
# ================================================================
# CELL 3 — PIPELINE CONFIGURATION
# ================================================================

CATALOG = "genai_copilot"

BRONZE_TABLE = f"{CATALOG}.bronze.sales_raw"
SILVER_TABLE = f"{CATALOG}.silver.sales"
GOLD_TABLE = f"{CATALOG}.gold.region_sales"

print("=" * 70)
print("PIPELINE CONFIGURATION")
print("=" * 70)

print("Catalog:", CATALOG)
print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)
print("Gold:", GOLD_TABLE)

In [0]:
# ================================================================
# CELL 4 — PIPELINE STAGES
# ================================================================

pipeline_stages = [
    {
        "stage_id": 1,
        "name": "bronze_ingestion",
        "description": "Load raw sales data into Bronze",
        "notebook": "02_data_pipeline.py"
    },
    {
        "stage_id": 2,
        "name": "silver_transformation",
        "description": "Clean and transform Bronze data",
        "notebook": "02_data_pipeline.py"
    },
    {
        "stage_id": 3,
        "name": "gold_aggregation",
        "description": "Create analytical Gold tables",
        "notebook": "02_data_pipeline.py"
    },
    {
        "stage_id": 4,
        "name": "data_quality",
        "description": "Validate pipeline data quality",
        "notebook": "02_data_pipeline.py"
    },
    {
        "stage_id": 5,
        "name": "rag_pipeline",
        "description": "Build and refresh RAG knowledge pipeline",
        "notebook": "03_rag_pipeline.py"
    },
    {
        "stage_id": 6,
        "name": "copilot_pipeline",
        "description": "Validate SQL, RAG and Hybrid copilot",
        "notebook": "04_copilot_pipeline.py"
    }
]

print("=" * 70)
print("PIPELINE STAGES")
print("=" * 70)

for stage in pipeline_stages:

    print(
        f"{stage['stage_id']}. "
        f"{stage['name']} → "
        f"{stage['description']}"
    )

In [0]:
# ================================================================
# CELL 5 — PIPELINE DEPENDENCIES
# ================================================================

pipeline_dependencies = {
    "bronze_ingestion": [],
    "silver_transformation": [
        "bronze_ingestion"
    ],
    "gold_aggregation": [
        "silver_transformation"
    ],
    "data_quality": [
        "gold_aggregation"
    ],
    "rag_pipeline": [
        "data_quality"
    ],
    "copilot_pipeline": [
        "rag_pipeline"
    ]
}

print("=" * 70)
print("PIPELINE DEPENDENCIES")
print("=" * 70)

for stage, dependencies in pipeline_dependencies.items():

    print(
        f"{stage:25} <- "
        f"{dependencies if dependencies else 'START'}"
    )

In [0]:
# ================================================================
# CELL 6 — DEPENDENCY VALIDATION
# ================================================================

print("=" * 70)
print("DEPENDENCY VALIDATION")
print("=" * 70)

stage_names = [
    stage["name"]
    for stage in pipeline_stages
]

dependency_validation_passed = True

for stage in pipeline_stages:

    stage_name = stage["name"]

    dependencies = pipeline_dependencies.get(
        stage_name,
        []
    )

    stage_position = stage_names.index(
        stage_name
    )

    for dependency in dependencies:

        if dependency not in stage_names:

            print(
                f"FAIL - Unknown dependency: "
                f"{dependency}"
            )

            dependency_validation_passed = False

        else:

            dependency_position = stage_names.index(
                dependency
            )

            if dependency_position >= stage_position:

                print(
                    f"FAIL - Invalid order: "
                    f"{dependency} → {stage_name}"
                )

                dependency_validation_passed = False

print()

print(
    "Dependency validation:",
    "PASS"
    if dependency_validation_passed
    else "FAIL"
)

if not dependency_validation_passed:

    raise RuntimeError(
        "Pipeline dependency validation failed."
    )

In [0]:
# ================================================================
# CELL 7 — SCHEDULING CONFIGURATION
# ================================================================

schedule_config = {
    "pipeline_name": "genai_data_analyst_copilot_pipeline",

    "schedule_type": "daily",

    "schedule_time": "02:00",

    "timezone": "Asia/Tokyo",

    "enabled": False,

    "mode": "validation"
}

print("=" * 70)
print("SCHEDULING CONFIGURATION")
print("=" * 70)

print(
    json.dumps(
        schedule_config,
        indent=2
    )
)

Important: enabled = False intentionally. We are validating the design before creating a real scheduled job.

In [0]:
# ================================================================
# CELL 8 — RETRY CONFIGURATION
# ================================================================

retry_config = {
    "max_retries": 2,
    "retry_interval_minutes": 5,
    "timeout_minutes": 60
}

print("=" * 70)
print("RETRY CONFIGURATION")
print("=" * 70)

print(
    json.dumps(
        retry_config,
        indent=2
    )
)

In [0]:
# ================================================================
# CELL 10 — JOB DEFINITION
# ================================================================

job_definition = {
    "name": schedule_config["pipeline_name"],

    "schedule": {
        "type": schedule_config["schedule_type"],
        "time": schedule_config["schedule_time"],
        "timezone": schedule_config["timezone"],
        "enabled": schedule_config["enabled"]
    },

    "tasks": pipeline_stages,

    "dependencies": pipeline_dependencies,

    "retry_policy": retry_config,

    # "failure_policy": failure_policy
}

print("=" * 70)
print("JOB DEFINITION")
print("=" * 70)

print(
    json.dumps(
        job_definition,
        indent=2,
        default=str
    )
)

In [0]:
# ================================================================
# CELL 12 — STAGE VALIDATION
# ================================================================

print("=" * 70)
print("PIPELINE STAGE VALIDATION")
print("=" * 70)

stage_validation_passed = True

for stage in pipeline_stages:

    required_fields = [
        "stage_id",
        "name",
        "description",
        "notebook"
    ]

    missing = [
        field
        for field in required_fields
        if field not in stage
    ]

    if missing:

        stage_validation_passed = False

        print(
            f"FAIL - {stage.get('name', 'UNKNOWN')}: "
            f"missing {missing}"
        )

    else:

        print(
            f"PASS - {stage['name']}"
        )

print()

if not stage_validation_passed:

    raise RuntimeError(
        "One or more pipeline stages are invalid."
    )

print("Stage validation: PASS")

In [0]:
# ================================================================
# CELL 13 — DRY-RUN EXECUTION PLAN
# ================================================================

print("=" * 70)
print("DRY-RUN EXECUTION PLAN")
print("=" * 70)

print()

for index, stage in enumerate(
    pipeline_stages,
    start=1
):

    dependencies = pipeline_dependencies.get(
        stage["name"],
        []
    )

    print(
        f"STEP {index}: "
        f"{stage['name']}"
    )

    print(
        f"       Notebook: "
        f"{stage['notebook']}"
    )

    print(
        f"       Depends on: "
        f"{dependencies if dependencies else 'START'}"
    )

    print()

In [0]:
# ================================================================
# CELL 14 — SCHEDULING SAFETY CHECK
# ================================================================

print("=" * 70)
print("SCHEDULING SAFETY CHECK")
print("=" * 70)

safety_checks = [
    (
        "Schedule configuration exists",
        bool(schedule_config)
    ),
    (
        "Pipeline schedule disabled during validation",
        schedule_config["enabled"] is False
    ),
    (
        "Retry configuration exists",
        bool(retry_config)
    ),
    # (
    #     "Failure policy exists",
    #     bool(failure_policy)
    # ),
    (
        "Pipeline stages defined",
        len(pipeline_stages) > 0
    ),
    (
        "Dependencies defined",
        len(pipeline_dependencies) > 0
    )
]

safety_failures = 0

for check_name, passed in safety_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{check_name}"
    )

    if not passed:
        safety_failures += 1

print()

print(
    "Safety checks:",
    len(safety_checks)
)

print(
    "Failed checks:",
    safety_failures
)

if safety_failures:

    raise RuntimeError(
        "Scheduling safety validation failed."
    )

print()
print("Scheduling safety: PASS")

In [0]:
# ================================================================
# CELL 15 — SCHEDULING METADATA
# ================================================================

scheduling_metadata = {
    "pipeline_name": schedule_config["pipeline_name"],
    "phase": 22,
    "status": "validated",
    "schedule_enabled": schedule_config["enabled"],
    "schedule_type": schedule_config["schedule_type"],
    "schedule_time": schedule_config["schedule_time"],
    "timezone": schedule_config["timezone"],
    "task_count": len(pipeline_stages),
    "retry_policy": retry_config,
    # "failure_policy": failure_policy,
    "validated_at": datetime.now().isoformat()
}

print("=" * 70)
print("SCHEDULING METADATA")
print("=" * 70)

print(
    json.dumps(
        scheduling_metadata,
        indent=2,
        default=str
    )
)